# Adaptive Graph Engine: Colab + Kaggle Runbook

End-to-end setup and execution on **Google Colab** or **Kaggle Notebooks**.

In [45]:
# Run this as a new cell in your notebook
import json

with open('/kaggle/working/__notebook__.ipynb', 'r') as f:
    nb = json.load(f)

for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        src = ''.join(cell['source'])
        print(f"\n{'='*60}")
        print(f"CELL {i} (code, {len(src)} chars)")
        print('='*60)
        print(src)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/__notebook__.ipynb'

In [2]:
# Detect environment and print basics
import os, sys, platform, subprocess, shutil

is_colab = 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules
is_kaggle = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')

ENV = "colab" if is_colab else "kaggle" if is_kaggle else "local"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Environment:", ENV)

# -------------------------------
# GPU CHECK (SAFE)
# -------------------------------
print("\n=== GPU Info ===")

if shutil.which("nvidia-smi") is not None:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(gpu.stdout)
    has_gpu = True
else:
    print("❌ nvidia-smi not found → GPU likely not enabled")
    has_gpu = False

# -------------------------------
# NVCC CHECK
# -------------------------------
print("\n=== NVCC Check ===")

if shutil.which("nvcc") is not None:
    try:
        nvcc_out = subprocess.check_output(["nvcc", "--version"], text=True)
        print(nvcc_out.split("\n")[-2])
        has_nvcc = True
    except:
        print("⚠️ nvcc exists but failed to run")
        has_nvcc = False
else:
    print("❌ nvcc not found")
    has_nvcc = False

# -------------------------------
# SUMMARY
# -------------------------------
print("\n=== Summary ===")
print("GPU:", "Yes" if has_gpu else "No")
print("NVCC:", "Yes" if has_nvcc else "No")

# -------------------------------
# CONFIG OBJECT (SHARED STATE)
# -------------------------------
CONFIG = {
    "env": ENV,
    "has_gpu": has_gpu,
    "has_nvcc": has_nvcc
}

Python: 3.12.12
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Environment: kaggle

=== GPU Info ===
Sat Apr 11 12:43:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                      

In [3]:
# System dependencies for build

print("=== Installing system dependencies ===")

!apt-get update -y
!apt-get install -y cmake libgomp1 build-essential git

# -------------------------------
# CMAKE CHECK
# -------------------------------
print("\n=== CMake Version ===")
!cmake --version

# -------------------------------
# CUDA INSTALL (SMART)
# -------------------------------
print("\n=== CUDA Setup ===")

if not CONFIG["has_gpu"]:
    print("⚠️ No GPU detected → Skipping CUDA installation")
    print("➡️ System will run in SEQ / OpenMP mode only")

elif not CONFIG["has_nvcc"]:
    print("⚠️ NVCC missing but GPU available → Installing CUDA toolkit...")
    !apt-get install -y cuda-toolkit-11-8

else:
    print("✅ CUDA already available")

# -------------------------------
# FINAL CHECK
# -------------------------------
print("\n=== NVCC Version ===")
!nvcc --version || echo "❌ nvcc still not available"

=== Installing system dependencies ===
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [88.5 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,497 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launch

## Repository Setup
If notebook is outside your repo, set `REPO_URL` and run clone cell.
If notebook is already in repo root, keep `REPO_URL` empty.

In [4]:
import os, shutil

REPO_URL = "https://github.com/SamvedSama/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis.git"
BRANCH   = "Sama"

if ENV == "colab":
    BASE_DIR = "/content"
elif ENV == "kaggle":
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = os.getcwd()

REPO_DIR = os.path.join(BASE_DIR, "Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis")

# 🔥 FIX: move out before deleting
os.chdir(BASE_DIR)

# Always wipe and reclone
if os.path.exists(REPO_DIR):
    print(f"Removing existing clone at {REPO_DIR}...")
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch '{BRANCH}'...")
clone_status = os.system(f"git clone -b {BRANCH} {REPO_URL} {REPO_DIR}")

if clone_status != 0:
    raise RuntimeError("❌ Git clone failed — check REPO_URL and BRANCH")

WORKDIR = REPO_DIR
print("WORKDIR =", WORKDIR)

%cd {WORKDIR}

print("=== Latest commit ===")
!git log --oneline -5

print("=== Repo Contents ===")
!ls -la

Cloning branch 'Sama'...


Cloning into '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis'...


WORKDIR = /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis
/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis
=== Latest commit ===
63b05ec (HEAD -> Sama, origin/Sama) Update pagerank_cuda.cu
c1924d3 Update wcc_cuda.cu
4d60d83 Refactor WCC algorithm for improved correctness and performance
0bd4c5a Update wcc_seq.cpp
ad3b29a Update bfs_cuda.cu
=== Repo Contents ===
total 60
drwxr-xr-x 8 root root 4096 Apr 11 12:43 .
drwxr-xr-x 4 root root 4096 Apr 11 12:43 ..
drwxr-xr-x 3 root root 4096 Apr 11 12:43 benchmarks
-rw-r--r-- 1 root root 3890 Apr 11 12:43 CMakeLists.txt
-rw-r--r-- 1 root root 5111 Apr 11 12:43 Colab_Kaggle_Runbook.ipynb
drwxr-xr-x 2 root root 4096 Apr 11 12:43 data
drwxr-xr-x 8 root root 4096 Apr 11 12:43 .git
drwxr-xr-x 2 root root 4096 Apr 11 12:43 include
-rw-r--r-- 1 root root 2426 Apr 11 12:43 Makefile
drwxr-xr-x 2 root root 4096 Apr 11 12:43 python
-rw-r--r-- 1 root root 5291 Apr 11 12:43 R

In [5]:
# Python dependencies setup

import os

print("=== Installing Python dependencies ===")

# -------------------------------
# Upgrade pip
# -------------------------------
!python -m pip install --upgrade pip

# -------------------------------
# Install requirements
# -------------------------------
if os.path.exists("requirements.txt"):
    print("\nInstalling from requirements.txt...")
    !python -m pip install -r requirements.txt
else:
    print("⚠️ requirements.txt not found")

# -------------------------------
# Verify installations
# -------------------------------
print("\n=== Verifying packages ===")

try:
    import matplotlib
    import pandas
    import networkx
    
    print("✅ matplotlib:", matplotlib.__version__)
    print("✅ pandas:", pandas.__version__)
    print("✅ networkx:", networkx.__version__)
    
except Exception as e:
    print("❌ Verification failed:", e)

=== Installing Python dependencies ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

Installing from requirements.txt...

=== Verifying packages ===
✅ matplotlib: 3.10.0
✅ pandas: 2.3.3
✅ networkx: 3.6.1


In [6]:
!bash data/download.sh
!ls -lh data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  213k  100  213k    0     0   260k      0 --:--:-- --:--:-- --:--:--  260k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10.1M  100 10.1M    0     0  5010k      0  0:00:02  0:00:02 --:--:-- 5011k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  380M  100  380M    0     0  26.5M      0  0:00:14  0:00:14 --:--:-- 42.8M
Extracting facebook_combined.txt.gz
Extracting twitter_combined.txt.gz
Extracting gplus_combined.txt.gz
Done. Edge lists are in: /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data
total 1.3G
-rw-r--r-- 1 root root  920 Apr 11 12:43 download.sh
-

In [7]:
!python python/preprocess.py data/facebook_combined.txt -o data/processed/facebook_combined.txt
!python python/preprocess.py data/twitter_combined.txt  -o data/processed/twitter_combined.txt
!python python/preprocess.py data/gplus_combined.txt    -o data/processed/gplus_combined.txt
!ls -lh data/processed

Input: data/facebook_combined.txt
Output: data/processed/facebook_combined.txt
Undirected hint in comments: False
N (nodes) = 4039
M (unique edges after remap, no self-loops) = 88234
Density (undirected approx) = 1.081996e-02
Input: data/twitter_combined.txt
Output: data/processed/twitter_combined.txt
Undirected hint in comments: False
N (nodes) = 81306
M (unique edges after remap, no self-loops) = 1768135
Density (undirected approx) = 5.349406e-04
Input: data/gplus_combined.txt
Output: data/processed/gplus_combined.txt
Undirected hint in comments: False
N (nodes) = 107614
M (unique edges after remap, no self-loops) = 13673453
Density (undirected approx) = 2.361428e-03
total 169M
-rw-r--r-- 1 root root 820K Apr 11 12:44 facebook_combined.txt
-rw-r--r-- 1 root root 149M Apr 11 12:45 gplus_combined.txt
-rw-r--r-- 1 root root  20M Apr 11 12:44 twitter_combined.txt


In [8]:
!cd data && ls && head facebook_combined.txt && head twitter_combined.txt

download.sh	       gplus_combined.txt  README.md
facebook_combined.txt  processed	   twitter_combined.txt
0 1
0 2
0 3
0 4
0 5
0 6
0 7
0 8
0 9
0 10
214328887 34428380
17116707 28465635
380580781 18996905
221036078 153460275
107830991 17868918
151338729 222261763
19705747 34428380
222261763 88323281
19933035 149538028
158419434 17434613


In [9]:
# Pre-build source verification — confirm GitHub has the correct fixes
# If any assert fires, the file on GitHub still has old content.
import os
import sys

SRC = os.path.join(WORKDIR, "src/algorithms")

checks = [
    # (file_relative_path, string_that_MUST_exist, string_that_must_NOT_exist)
    (
        "bfs/bfs_cuda.cu",
        "csr.row_ptr.data()",  # fixed
        "std::copy(csr.row_ptr, csr.row_ptr +"  # old broken version
    ),
    (
        "wcc/wcc_omp.cpp",
        "NodeID find_root(std::vector<std::atomic<NodeID>>& parent",  # fixed (non-const)
        "NodeID find_root(const std::vector<std::atomic<NodeID>>& parent"  # old broken
    ),
    (
        "wcc/wcc_omp.cpp",
        "NodeID expected = p;",  # fixed
        "const_cast<NodeID&>(p)"  # old broken
    ),
]

all_ok = True

for rel_path, must_have, must_not_have in checks:
    full_path = os.path.join(SRC, rel_path)

    with open(full_path) as f:
        content = f.read()

    ok_have = must_have in content
    ok_not_have = must_not_have not in content

    status = "✅" if (ok_have and ok_not_have) else "❌"
    print(f"{status} {rel_path}")

    if not ok_have:
        print(f"   MISSING:  {must_have!r}")
        all_ok = False

    if not ok_not_have:
        print(f"   STILL HAS: {must_not_have!r}")
        all_ok = False


if not all_ok:
    print()
    print("❌ GitHub still has unfixed files. Apply patches below, then rebuild.")
    print("   Auto-patching now...")

    # ---- Patch BFS ----
    bfs_path = os.path.join(SRC, "bfs/bfs_cuda.cu")

    with open(bfs_path) as f:
        src = f.read()

    src = (
        src.replace(
            "std::copy(csr.row_ptr, csr.row_ptr + row_count, row_host.begin());",
            "std::copy(csr.row_ptr.data(), csr.row_ptr.data() + row_count, row_host.begin());"
        )
        .replace(
            "std::copy(csr.col_idx, csr.col_idx + col_count, col_host.begin());",
            "std::copy(csr.col_idx.data(), csr.col_idx.data() + col_count, col_host.begin());"
        )
    )

    with open(bfs_path, "w") as f:
        f.write(src)

    print("   ✅ bfs_cuda.cu patched")

    # ---- Patch WCC ----
    wcc_path = os.path.join(SRC, "wcc/wcc_omp.cpp")

    with open(wcc_path) as f:
        src = f.read()

    src = (
        src.replace(
            "NodeID find_root(const std::vector<std::atomic<NodeID>>& parent, NodeID x)",
            "NodeID find_root(std::vector<std::atomic<NodeID>>& parent, NodeID x)"
        )
        .replace(
            """        parent[x].compare_exchange_weak(
            const_cast<NodeID&>(p), gp,
            std::memory_order_relaxed, std::memory_order_relaxed);""",
            """        NodeID expected = p;
        parent[x].compare_exchange_weak(
            expected, gp,
            std::memory_order_relaxed, std::memory_order_relaxed);"""
        )
    )

    with open(wcc_path, "w") as f:
        f.write(src)

    print("   ✅ wcc_omp.cpp patched")
    print()
    print("Build will proceed with patched files.")

else:
    print()
    print("✅ All source files verified — safe to build.")

❌ bfs/bfs_cuda.cu
   MISSING:  'csr.row_ptr.data()'
   STILL HAS: 'std::copy(csr.row_ptr, csr.row_ptr +'
❌ wcc/wcc_omp.cpp
   MISSING:  'NodeID find_root(std::vector<std::atomic<NodeID>>& parent'
   STILL HAS: 'NodeID find_root(const std::vector<std::atomic<NodeID>>& parent'
❌ wcc/wcc_omp.cpp
   MISSING:  'NodeID expected = p;'
   STILL HAS: 'const_cast<NodeID&>(p)'

❌ GitHub still has unfixed files. Apply patches below, then rebuild.
   Auto-patching now...
   ✅ bfs_cuda.cu patched
   ✅ wcc_omp.cpp patched

Build will proceed with patched files.


In [10]:
!mkdir -p build
%cd build
!cmake ..
!cmake --build . -j4
!ls -lh

/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/build
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "12.8.93")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP_CUDA: -fopenmp (found version "4.5")
-- Found OpenMP: 

In [11]:
%%writefile /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/download.sh
#!/usr/bin/env bash
set -euo pipefail

ROOT="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
cd "$ROOT"

fetch() {
  local url="$1"
  local out="$2"

  echo "Downloading $url"
  if command -v curl >/dev/null 2>&1; then
    curl -L --fail -o "$out" "$url"
  else
    wget -O "$out" "$url"
  fi
}

# Small datasets
fetch "https://snap.stanford.edu/data/facebook_combined.txt.gz" "facebook_combined.txt.gz"
fetch "https://snap.stanford.edu/data/twitter_combined.txt.gz" "twitter_combined.txt.gz"

# Medium + Large datasets
fetch "https://snap.stanford.edu/data/soc-Slashdot0811.txt.gz" "slashdot.txt.gz"
fetch "https://snap.stanford.edu/data/soc-LiveJournal1.txt.gz" "livejournal.txt.gz"

# Extract
for gz in *.txt.gz; do
  echo "Extracting $gz"
  gunzip -f "$gz"
done

# Clean (remove comments)
for file in *.txt; do
  echo "Cleaning $file"
  grep -v "^#" "$file" > "${file%.txt}_clean.txt"
done

echo "Done. Files are in $ROOT"

Overwriting /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/download.sh


In [44]:
# Run this cell in your notebook
import os

files_to_print = [
    "CMakeLists.txt",
    "src/main.cpp",
    "src/scheduler/scheduler.h",
    "src/scheduler/scheduler.cpp",
    "src/graph/graph.h",
    "src/graph/graph.cpp",
    "src/graph/loader.cpp",
    "src/algorithms/bfs/bfs_cuda.cu",
    "src/algorithms/pagerank/pagerank_cuda.cu",
    "src/algorithms/wcc/wcc_cuda.cu",
    "benchmarks/benchmark_runner.cpp",
    "python/preprocess.py",
    "python/visualize.py",
]

BASE = "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis"  # adjust if different

for f in files_to_print:
    full = os.path.join(BASE, f)
    print(f"\n{'='*60}")
    print(f"FILE: {f}")
    print('='*60)
    if os.path.exists(full):
        with open(full) as fh:
            print(fh.read())
    else:
        print(f"NOT FOUND at {full}")


FILE: CMakeLists.txt
# Top-level build for the adaptive heterogeneous graph engine (C++17 + CUDA + OpenMP).
cmake_minimum_required(VERSION 3.18)

project(adaptive_graph_engine LANGUAGES CXX CUDA)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

set(CMAKE_CUDA_STANDARD 17)
set(CMAKE_CUDA_STANDARD_REQUIRED ON)
set(CMAKE_CUDA_ARCHITECTURES 75)

# ── CUDAToolkit MUST come before any target_link_libraries that reference
#    CUDA::cudart. find_package(CUDAToolkit) is what creates that imported target.
find_package(CUDAToolkit REQUIRED)
find_package(OpenMP REQUIRED)

# ── Collect the CUDA include path so g++-compiled .cpp files that transitively
#    include common.h (which guards cuda_runtime.h behind GRAPH_ENGINE_BUILD_CUDA)
#    can still locate the header when the definition is set.
#    CUDAToolkit_INCLUDE_DIRS is populated by find_package(CUDAToolkit).
set(ENGINE_INCLUDES
    ${CMAKE_SOURCE_DIR}/include
    ${CMAKE_SOURCE_DIR}/src
    ${CU

In [13]:
!cd /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data && chmod +x download.sh && ./download.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  213k  100  213k    0     0   259k      0 --:--:-- --:--:-- --:--:--  259k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10.1M  100 10.1M    0     0  3348k      0  0:00:03  0:00:03 --:--:-- 3348k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3024k  100 3024k    0     0  1637k      0  0:00:01  0:00:01 --:--:-- 1637k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  247M  100  247M    0     0  22.7M      0  0:00:10  0:00:10 --:--:-- 41.5M
Extracting facebook_combined.txt.gz
Extracting livej

In [14]:
!mkdir -p data/processed

In [15]:
%cd /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis
!python python/preprocess.py data/facebook_combined.txt -o data/processed/facebook_combined.txt

!python python/preprocess.py data/twitter_combined.txt -o data/processed/twitter_combined.txt

!python python/preprocess.py data/gplus_combined.txt -o data/processed/gplus_combined.txt

!python python/preprocess.py data/slashdot.txt -o data/processed/slashdot.txt

!python python/preprocess.py data/livejournal.txt -o data/processed/livejournal.txt

!ls -lh data/processed

/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis
Input: data/facebook_combined.txt
Output: data/processed/facebook_combined.txt
Undirected hint in comments: False
N (nodes) = 4039
M (unique edges after remap, no self-loops) = 88234
Density (undirected approx) = 1.081996e-02
Input: data/twitter_combined.txt
Output: data/processed/twitter_combined.txt
Undirected hint in comments: False
N (nodes) = 81306
M (unique edges after remap, no self-loops) = 1768135
Density (undirected approx) = 5.349406e-04
^C
Traceback (most recent call last):
  File "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/python/preprocess.py", line 108, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/python/preprocess.py", line 85, in main
    edges, hint = load_edges(inp)
                  ^^^^^^^^^^^^^^^
  File "/kagg

In [16]:
DATASET = 'facebook'  # facebook | twitter | gplus
GRAPH = f'../data/processed/{DATASET}_combined.txt'
print('GRAPH =', GRAPH)

GRAPH = ../data/processed/facebook_combined.txt


In [17]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto | tee ../engine_auto_facebook.log

/bin/bash: line 1: ./graph_engine: No such file or directory


In [18]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto --compare | tee ../engine_compare_facebook.log

/bin/bash: line 1: ./graph_engine: No such file or directory


In [19]:
TRIALS = 3
CSV_PATH = "../benchmarks/results/results.csv"
!./benchmark_runner --graph "{GRAPH}" --dataset "{DATASET}" --trials {TRIALS} --outcsv "{CSV_PATH}" | tee ../benchmark_facebook.log

/bin/bash: line 1: ./benchmark_runner: No such file or directory


In [20]:
DATASET = 'twitter'  # facebook | twitter | gplus
GRAPH = f'../data/processed/{DATASET}_combined.txt'
print('GRAPH =', GRAPH)

GRAPH = ../data/processed/twitter_combined.txt


In [21]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto | tee ../engine_auto_twitter.log

/bin/bash: line 1: ./graph_engine: No such file or directory


In [22]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto --compare | tee ../engine_compare_twitter.log

/bin/bash: line 1: ./graph_engine: No such file or directory


In [23]:
TRIALS = 3
CSV_PATH = "../benchmarks/results/results.csv"
!./benchmark_runner --graph "{GRAPH}" --dataset "{DATASET}" --trials {TRIALS} --outcsv "{CSV_PATH}" | tee ../benchmark_twitter.log

/bin/bash: line 1: ./benchmark_runner: No such file or directory


In [24]:
DATASET = 'gplus'  # facebook | twitter | gplus
GRAPH = f'../data/processed/{DATASET}_combined.txt'
print('GRAPH =', GRAPH)

GRAPH = ../data/processed/gplus_combined.txt


In [25]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 5 --mode auto | tee ../engine_auto_gplus.log

/bin/bash: line 1: ./graph_engine: No such file or directory


In [26]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto --compare | tee ../engine_compare_gplus.log
# gplus is too large for --compare (all 3 backends × all algos OOM/times out)
# Run auto-mode only and copy log as the compare log too
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 5 --mode auto | tee ../engine_compare_gplus.log

/bin/bash: line 1: ./graph_engine: No such file or directory
/bin/bash: line 1: ./graph_engine: No such file or directory


In [27]:
TRIALS = 3
CSV_PATH = "../benchmarks/results/results.csv"
!./benchmark_runner --graph "{GRAPH}" --dataset "{DATASET}" --trials {TRIALS} --outcsv "{CSV_PATH}" | tee ../benchmark_gplus.log
!python ../python/visualize.py --csv {CSV_PATH} --outdir ../benchmarks/results
!ls -lh ../benchmarks/results

/bin/bash: line 1: ./benchmark_runner: No such file or directory
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory
ls: cannot access '../benchmarks/results': No such file or directory


In [28]:
!python ../python/analytics_report.py --log ../engine_compare_facebook.log
!python ../python/analytics_report.py --log ../engine_compare_twitter.log
!python ../python/analytics_report.py --log ../engine_compare_gplus.log

python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/analytics_report.py': [Errno 2] No such file or directory
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/analytics_report.py': [Errno 2] No such file or directory
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/analytics_report.py': [Errno 2] No such file or directory


In [29]:
DATA_DIR = "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data"

DATASETS = {
    # 🔹 LOW
    "facebook": f"{DATA_DIR}/facebook_combined.txt",
    
    # 🔹 MEDIUM
    "twitter": f"{DATA_DIR}/twitter_combined.txt", 
    "slashdot": f"{DATA_DIR}/slashdot_clean.txt",
    
    # 🔹 HIGH
    "livejournal": f"{DATA_DIR}/livejournal_clean.txt",
}

In [30]:
import os

for name, path in DATASETS.items():
    print(f"{name}: {'✅' if os.path.exists(path) else '❌'} -> {path}")

facebook: ✅ -> /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/facebook_combined.txt
twitter: ✅ -> /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/twitter_combined.txt
slashdot: ✅ -> /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/slashdot_clean.txt
livejournal: ✅ -> /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/livejournal_clean.txt


In [31]:
import os

ENGINE = "./graph_engine"
LOG_DIR = "/kaggle/working/logs"

os.makedirs(LOG_DIR, exist_ok=True)

for name, graph in DATASETS.items():
    print(f"\n🚀 Running AUTO mode on {name.upper()}...\n")
    
    log_file = f"{LOG_DIR}/auto_{name}.log"
    
    cmd = f"""
    {ENGINE} \
    --graph "{graph}" \
    --algorithm all \
    --source 0 \
    --topk 10 \
    --mode auto | tee {log_file}
    """
    
    os.system(cmd)


🚀 Running AUTO mode on FACEBOOK...


🚀 Running AUTO mode on TWITTER...


🚀 Running AUTO mode on SLASHDOT...


🚀 Running AUTO mode on LIVEJOURNAL...



sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found


In [32]:
!find . -type f -executable -name "graph_engine"

./build/graph_engine


In [33]:
MODES = ["seq", "omp", "cuda"]

for name, graph in DATASETS.items():
    print(f"\n================ {name.upper()} ================\n")
    
    for mode in MODES:
        print(f"\n⚙️ Mode: {mode.upper()}\n")
        
        cmd = f"""
        ./graph_engine \
        --graph "{graph}" \
        --algorithm all \
        --source 0 \
        --topk 10 \
        --mode {mode}
        """
        
        os.system(cmd)


================ FACEBOOK ================


⚙️ Mode: SEQ


⚙️ Mode: OMP


⚙️ Mode: CUDA


================ TWITTER ================


⚙️ Mode: SEQ


⚙️ Mode: OMP


⚙️ Mode: CUDA


================ SLASHDOT ================


⚙️ Mode: SEQ


⚙️ Mode: OMP


⚙️ Mode: CUDA


================ LIVEJOURNAL ================


⚙️ Mode: SEQ


⚙️ Mode: OMP


⚙️ Mode: CUDA



sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found


In [34]:
import re

def extract_summary(log_file):
    with open(log_file) as f:
        lines = f.readlines()
    
    for line in lines:
        if "[scheduler]" in line or "speedup summary" in line:
            print(line.strip())

for name in DATASETS:
    print(f"\n📊 Summary for {name.upper()}")
    extract_summary(f"{LOG_DIR}/auto_{name}.log")


📊 Summary for FACEBOOK

📊 Summary for TWITTER

📊 Summary for SLASHDOT

📊 Summary for LIVEJOURNAL


In [35]:
import os

# -----------------------------
# CONFIG
# -----------------------------
DATA_DIR = "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data"
RESULTS_DIR = "../benchmarks/results"
LOG_DIR = "../logs"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

TRIALS = 3

DATASETS = {
    "facebook": f"{DATA_DIR}/facebook_combined.txt",
    "gplus": f"{DATA_DIR}/gplus_combined.txt",
    "slashdot": f"{DATA_DIR}/slashdot_clean.txt",
    # "livejournal": f"{DATA_DIR}/livejournal_clean.txt",  # optional (heavy)
}

MODES = ["seq", "omp", "cuda"]

# -----------------------------
# BENCHMARK RUNNER (CSV + analytics)
# -----------------------------
for DATASET, GRAPH in DATASETS.items():
    print(f"\n📊 BENCHMARK: {DATASET.upper()}\n")

    CSV_PATH = f"{RESULTS_DIR}/{DATASET}_results.csv"
    LOG_PATH = f"{LOG_DIR}/benchmark_{DATASET}.log"

    cmd = f"""
    ./benchmark_runner \
    --graph "{GRAPH}" \
    --dataset "{DATASET}" \
    --trials {TRIALS} \
    --outcsv "{CSV_PATH}" | tee {LOG_PATH}
    """
    os.system(cmd)

    # Generate plots (analytics)
    os.system(f"python ../python/visualize.py --csv {CSV_PATH} --outdir {RESULTS_DIR}")


📊 BENCHMARK: FACEBOOK



sh: 2: ./benchmark_runner: not found



📊 BENCHMARK: GPLUS



python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory
sh: 2: ./benchmark_runner: not found



📊 BENCHMARK: SLASHDOT



python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory
sh: 2: ./benchmark_runner: not found
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory


In [36]:
import os
import shutil

RESULTS_DIR = "../benchmarks/results"
LOG_DIR = "../logs"

# Remove old results completely
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)

if os.path.exists(LOG_DIR):
    shutil.rmtree(LOG_DIR)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print("✅ Cleaned old results")

✅ Cleaned old results


In [37]:
DATA_DIR = "/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data"

DATASETS = {
    "facebook": f"{DATA_DIR}/facebook_combined.txt",
    "gplus": f"{DATA_DIR}/gplus_combined.txt",
    "slashdot": f"{DATA_DIR}/slashdot_clean.txt",
    # "livejournal": f"{DATA_DIR}/livejournal_clean.txt",  # optional heavy
}

TRIALS = 3
MODES = ["seq", "omp", "cuda"]

In [38]:
!grep -E '^[0-9]+[[:space:]][0-9]+$' \
/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/gplus_combined.txt \
> /kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/data/gplus_clean.txt

^C


In [39]:
DATASETS["gplus"] = f"{DATA_DIR}/gplus_clean.txt"

In [40]:
for DATASET, GRAPH in DATASETS.items():
    print(f"\n📊 BENCHMARK: {DATASET.upper()}\n")

    CSV_PATH = f"{RESULTS_DIR}/{DATASET}_results.csv"
    LOG_PATH = f"{LOG_DIR}/benchmark_{DATASET}.log"

    cmd = f"""
    ./benchmark_runner \
    --graph "{GRAPH}" \
    --dataset "{DATASET}" \
    --trials {TRIALS} \
    --outcsv "{CSV_PATH}" | tee {LOG_PATH}
    """

    os.system(cmd)

    # Generate analytics plots
    os.system(f"python ../python/visualize.py --csv {CSV_PATH} --outdir {RESULTS_DIR}")


📊 BENCHMARK: FACEBOOK


📊 BENCHMARK: GPLUS


📊 BENCHMARK: SLASHDOT



sh: 2: ./benchmark_runner: not found
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory
sh: 2: ./benchmark_runner: not found
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory
sh: 2: ./benchmark_runner: not found
python3: can't open file '/kaggle/working/Adaptive-Heterogeneous-Graph-Engine-for-Social-Network-Influence-Analysis/../python/visualize.py': [Errno 2] No such file or directory


In [41]:
for DATASET, GRAPH in DATASETS.items():
    print(f"\n⚙️ FULL ENGINE RUN: {DATASET.upper()}\n")

    for mode in MODES:
        print(f"\n🔹 Running {mode.upper()}...\n")

        LOG_PATH = f"{LOG_DIR}/engine_{DATASET}_{mode}.log"

        cmd = f"""
        ./graph_engine \
        --graph "{GRAPH}" \
        --algorithm all \
        --source 0 \
        --topk 10 \
        --mode {mode} \
        --compare | tee {LOG_PATH}
        """

        os.system(cmd)


⚙️ FULL ENGINE RUN: FACEBOOK


🔹 Running SEQ...


🔹 Running OMP...


🔹 Running CUDA...


⚙️ FULL ENGINE RUN: GPLUS


🔹 Running SEQ...


🔹 Running OMP...


🔹 Running CUDA...


⚙️ FULL ENGINE RUN: SLASHDOT


🔹 Running SEQ...


🔹 Running OMP...


🔹 Running CUDA...



sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found
sh: 2: ./graph_engine: not found


In [42]:
!ls -lh ../benchmarks/results
!ls -lh ../logs

total 0
total 0
-rw-r--r-- 1 root root 0 Apr 11 12:48 benchmark_facebook.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 benchmark_gplus.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 benchmark_slashdot.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_facebook_cuda.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_facebook_omp.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_facebook_seq.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_gplus_cuda.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_gplus_omp.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_gplus_seq.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_slashdot_cuda.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_slashdot_omp.log
-rw-r--r-- 1 root root 0 Apr 11 12:48 engine_slashdot_seq.log


In [43]:
# Merge CSV results and display + plot

import os
import pandas as pd
import matplotlib.pyplot as plt

results_dir = "../benchmarks/results"

# Collect all CSV files
csv_files = [f for f in os.listdir(results_dir) if f.endswith("_results.csv")]

dfs = []

for file in csv_files:
    path = os.path.join(results_dir, file)
    df = pd.read_csv(path)
    dataset_name = file.replace("_results.csv", "")
    df["dataset"] = dataset_name
    dfs.append(df)

# Merge all
merged_df = pd.concat(dfs, ignore_index=True)

# Show table
merged_df.head()

# Save merged CSV
merged_path = os.path.join(results_dir, "merged_results.csv")
merged_df.to_csv(merged_path, index=False)

# Plot: time comparison
plt.figure()
for dataset in merged_df["dataset"].unique():
    subset = merged_df[merged_df["dataset"] == dataset]
    plt.plot(subset["backend"], subset["time_ms"], marker='o', label=dataset)

plt.xlabel("Backend")
plt.ylabel("Time (ms)")
plt.title("Backend Time Comparison Across Datasets")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot: speedup comparison
plt.figure()
for dataset in merged_df["dataset"].unique():
    subset = merged_df[merged_df["dataset"] == dataset]
    plt.plot(subset["backend"], subset["speedup_vs_seq"], marker='o', label=dataset)

plt.xlabel("Backend")
plt.ylabel("Speedup vs SEQ")
plt.title("Speedup Comparison Across Datasets")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

merged_df

ValueError: No objects to concatenate